# KS0223: управление и сенсоры (Presentation)

Ноутбук для демо текущего стенда: Unity API + ROS2 bridge.
Здесь есть проверка API, кадр камеры, короткий заезд и live-управление.


In [1]:
%pip install -q requests numpy matplotlib ipywidgets


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import base64
import io
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "README.md").exists() and (candidate / "python").exists():
            return candidate
    raise RuntimeError("Не найден корень репозитория.")


ROOT = find_repo_root(Path.cwd())
PYTHON_DIR = ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

from sim_client.http_client import SimClient
from sim_client.ks0223 import Ks0223Command, parse_telemetry

BASE_URL = os.getenv("UAVSIM_BASE_URL", "http://127.0.0.1:8000")
RESET_CFG = {
    "seed": 1,
    "timeScale": 1.0,
    "selectedTrackId": "track.basic_arena.v1",
    "selectedVehicleId": "vehicle.ks0223.v1",
    "trackParams": [],
    "vehicleParams": [],
    "flags": [],
}

client = SimClient(BASE_URL, timeout_s=20.0)
print("ROOT:", ROOT)
print("API :", BASE_URL)


ROOT: <repo>
API : http://127.0.0.1:8000


In [3]:
API_READY = False
try:
    health = client.health()
    contract = client.get_contract()
    API_READY = True
    print("health:", health)
    print("simulator:", contract.get("simulatorName"), "version:", contract.get("contractVersion"))
    print("vehicles:", [v.get("deviceId") for v in contract.get("availableVehicles", [])])
    print("tracks  :", [t.get("trackId") for t in contract.get("availableTracks", [])])
except Exception as exc:
    print("API недоступен:", exc)
    print("Запусти Unity и нажми Play, затем повтори ячейку.")


API недоступен: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [Errno 61] Connection refused"))
Запусти Unity и нажми Play, затем повтори ячейку.


In [4]:
def step_pwm(left: float, right: float, brake: float = 0.0) -> dict[str, Any]:
    cmd = Ks0223Command(
        left_pwm_norm=float(left),
        right_pwm_norm=float(right),
        brake=float(brake),
    ).to_step_command()
    return client.step(cmd)


def frame_to_array(step_result: dict[str, Any]) -> np.ndarray | None:
    frame = step_result.get("frame") or {}
    b64 = frame.get("dataBase64") or ""
    if not b64:
        return None
    payload = base64.b64decode(b64)
    return mpimg.imread(io.BytesIO(payload), format="jpg")


def tf(telemetry: dict[str, str], key: str, default: float = float("nan")) -> float:
    try:
        return float(telemetry.get(key, default))
    except Exception:
        return default


In [5]:
if not API_READY:
    print("Пропуск: API недоступен.")
else:
    _ = client.reset(RESET_CFG)
    sample = step_pwm(0.0, 0.0, brake=1.0)
    telemetry = parse_telemetry(sample)
    img = frame_to_array(sample)

    print("speed m/s       :", round(float((sample.get("state") or {}).get("speed", 0.0)), 4))
    print("ultrasonic m    :", tf(telemetry, "sensor.ultrasonic.front.m"))
    print("battery voltage :", tf(telemetry, "power.battery.voltage_v"))
    print("line s3         :", tf(telemetry, "sensor.line_tracker.s3_norm"))

    if img is None:
        print("Кадр не получен. Проверь: Unity в Play и API доступен.")
    else:
        plt.figure(figsize=(7, 3.8))
        plt.imshow(img)
        plt.title("Front camera")
        plt.axis("off")
        plt.show()


Пропуск: API недоступен.


In [6]:
records = []
snapshots = []

if not API_READY:
    print("Пропуск заезда: API недоступен.")
else:
    # Короткий сценарий: разгон -> поворот -> стабилизация -> стоп
    profile = []
    profile += [(0.40, 0.40, 0.0)] * 24
    profile += [(0.24, 0.50, 0.0)] * 18
    profile += [(0.35, 0.35, 0.0)] * 16
    profile += [(0.0, 0.0, 0.8)] * 8

    _ = client.reset(RESET_CFG)

    for i, (left, right, brake) in enumerate(profile):
        step = step_pwm(left, right, brake)
        state = step.get("state") or {}
        telemetry = parse_telemetry(step)
        records.append(
            {
                "step": i,
                "speed": float(state.get("speed", 0.0)),
                "ultra": tf(telemetry, "sensor.ultrasonic.front.m"),
                "voltage": tf(telemetry, "power.battery.voltage_v"),
                "line_s3": tf(telemetry, "sensor.line_tracker.s3_norm"),
            }
        )
        if i in (0, 20, len(profile) - 1):
            image = frame_to_array(step)
            if image is not None:
                snapshots.append((i, image))

print("steps:", len(records), "snapshots:", len(snapshots))


Пропуск заезда: API недоступен.
steps: 0 snapshots: 0


In [7]:
if not records:
    print("Нет данных для графиков. Запусти Unity/Play и повтори ячейки выше.")
else:
    x = np.array([r["step"] for r in records], dtype=float)
    speed = np.array([r["speed"] for r in records], dtype=float)
    ultra = np.array([r["ultra"] for r in records], dtype=float)
    voltage = np.array([r["voltage"] for r in records], dtype=float)
    line_s3 = np.array([r["line_s3"] for r in records], dtype=float)

    fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
    axes[0, 0].plot(x, speed)
    axes[0, 0].set_title("Speed m/s")
    axes[0, 0].set_xlabel("step")

    axes[0, 1].plot(x, ultra)
    axes[0, 1].set_title("Ultrasonic m")
    axes[0, 1].set_xlabel("step")

    axes[1, 0].plot(x, voltage)
    axes[1, 0].set_title("Battery V")
    axes[1, 0].set_xlabel("step")

    axes[1, 1].plot(x, line_s3)
    axes[1, 1].set_title("Line tracker S3")
    axes[1, 1].set_xlabel("step")

    plt.show()

    if snapshots:
        fig, axs = plt.subplots(1, len(snapshots), figsize=(4 * len(snapshots), 3), constrained_layout=True)
        if len(snapshots) == 1:
            axs = [axs]
        for ax, (idx, image) in zip(axs, snapshots):
            ax.imshow(image)
            ax.set_title(f"step {idx}")
            ax.axis("off")
        plt.show()


Нет данных для графиков. Запусти Unity/Play и повтори ячейки выше.


## Live-управление (опционально)

Слайдеры отправляют один шаг команды PWM и сразу показывают ключевые сенсоры.


In [8]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception as exc:
    print("ipywidgets недоступен:", exc)
else:
    left = widgets.FloatSlider(description="left", min=-1.0, max=1.0, step=0.01, value=0.0)
    right = widgets.FloatSlider(description="right", min=-1.0, max=1.0, step=0.01, value=0.0)
    brake = widgets.FloatSlider(description="brake", min=0.0, max=1.0, step=0.01, value=0.0)
    out = widgets.Output()

    def send_step(_):
        with out:
            out.clear_output(wait=True)
            step = step_pwm(left.value, right.value, brake.value)
            telemetry = parse_telemetry(step)
            print(
                {
                    "speed": float((step.get("state") or {}).get("speed", 0.0)),
                    "ultrasonic_m": tf(telemetry, "sensor.ultrasonic.front.m"),
                    "voltage_v": tf(telemetry, "power.battery.voltage_v"),
                }
            )
            image = frame_to_array(step)
            if image is not None:
                plt.figure(figsize=(6, 3.2))
                plt.imshow(image)
                plt.axis("off")
                plt.show()

    btn = widgets.Button(description="Send Step", button_style="primary")
    btn.on_click(send_step)
    display(widgets.VBox([widgets.HBox([left, right, brake]), btn, out]))
